# 03b - Modifying the MetaSchedule Search Space

Mechanical recipe for changing MetaSchedule's search space in TVM 0.23:
list the default rules, modify the list, sanity-check the design space,
pass it to `tune_tir()`.

In [1]:
import tvm
from tvm import meta_schedule as ms
from tvm.meta_schedule import schedule_rule as SR
from tvm.meta_schedule.space_generator import PostOrderApply

from workloads import get_workload

mod = get_workload("matmul_1024")
target = tvm.target.Target("llvm --num-cores=8")

## 1. Where the search space comes from

```
   target          schedule rules          space generator          design space
 (e.g. "llvm")  ─►  (per-block)      ─►    (PostOrderApply)    ─►    (templates)
   IRModule
```

Rules define legal transforms and tunable knobs per block. `PostOrderApply` walks the
IRModule in post-DFS order and applies them to produce parametric *templates* (schedules
still containing `sample_perfect_tile` / `sample_categorical` calls). The search strategy
later picks concrete values.

**To change the search space, change the rule list.** That's it.

## 2. The default rules for `"llvm"`

In [ ]:
default_rules = ms.ScheduleRule.create("llvm")
for i, r in enumerate(default_rules):
    print(f"  {i}: {type(r).__name__}")

| Rule | Contribution |
|---|---|
| `ApplyCustomRule` | Hook for user rules (no effect by default) |
| `InlineConstantScalars` | Folds scalar-constant blocks |
| `AutoInline` | Fuses element-wise blocks into consumers |
| `AddRFactor` | rfactor variants for reductions |
| `MultiLevelTiling` | **Main source of knobs**: SSRSRS tile factors |
| `ParallelizeVectorizeUnroll` | Parallel/vectorize/unroll annotations + knobs |
| `RandomComputeLocation` | Randomizes `compute_at` for free blocks |

For single-block matmul, the load-bearing rules are `MultiLevelTiling` and
`ParallelizeVectorizeUnroll`. Source: `python/tvm/meta_schedule/schedule_rule/`.

## 3. Inspecting the design space without tuning

Cheapest check that a modification does what you expect: look at the template trace.

In [17]:
def show_design_space(mod, target, rules, label):
    space = PostOrderApply(
        sch_rules=rules,
        postprocs="from-target",
        mutator_probs="from-target",
    )
    ctx = ms.TuneContext(mod=mod, target=target, space_generator=space, task_name=label)
    templates = ctx.generate_design_space()
    print(f"[{label}] templates: {len(templates)}")
    for i, t in enumerate(templates):
        print(f" {i:0>2d} "+ "-" * 66)
        print(t.trace)
        print("-" * 70)
    return templates

In [ ]:
_ = show_design_space(mod, target, default_rules, "default")
# Seems they differ at reverse_compute_at

Look for:
- `sample_perfect_tile(loop, n=4, ...)` — one per tiled loop (the `S`/`R` factors).
- `sample_categorical(candidates=[...])` — discrete knobs (unroll depths, etc.).

Removing a rule should make its sampling calls disappear from the trace.

## 4. Modification A — remove `ParallelizeVectorizeUnroll`

In [ ]:
def without(rules, class_names):
    names = set(class_names)
    return [r for r in rules if type(r).__name__ not in names]

rules_no_pvu = without(default_rules, ["ParallelizeVectorizeUnroll"])
print("Kept:", [type(r).__name__ for r in rules_no_pvu])
print()
_ = show_design_space(mod, target, rules_no_pvu, "no-pvu")

Fewer `sample_categorical` calls at the end; tiling structure unchanged. Expected effect
under a fixed budget: significant GFLOPS drop (no vectorization / outer parallelism).

## 5. Modification B — restrict `MultiLevelTiling`

Replace a rule with a reconfigured instance. Default `max_innermost_factor=64`; shrinking
to 16 restricts the set of legal factorizations.

In [ ]:
def replace_rule(rules, class_name, new_rule):
    return [new_rule if type(r).__name__ == class_name else r for r in rules]

tight_mlt = SR.MultiLevelTiling(structure="SSRSRS", max_innermost_factor=16)  # <-- Modify the structure and you'll see changes
rules_tight = replace_rule(default_rules, "MultiLevelTiling", tight_mlt)
_ = show_design_space(mod, target, rules_tight, "tight-mlt")

Template shape unchanged — the restriction acts at sampling time, not visible in the trace.
Expected effect: faster convergence on a smaller space, but possibly lower peak if the best
factorization has been excluded.

## 6. Running `tune_tir()` with a custom space

Pass the `PostOrderApply` object as `space=`. Small budget here — real experiments belong
in standalone scripts.

In [ ]:
import os, shutil

WORK_DIR = "./tune_space_demo_no_pvu"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

space_no_pvu = PostOrderApply(
    sch_rules=rules_no_pvu, postprocs="from-target", mutator_probs="from-target",
)

db = ms.tune_tir(
    mod=mod, target=target, work_dir=WORK_DIR,
    max_trials_global=32, num_trials_per_iter=16,
    space=space_no_pvu, seed=42,
)

sch = db.query_schedule(mod, target, "main")
print(sch.trace)

## Recipe

1. `rules = ms.ScheduleRule.create("llvm")`
2. Modify with `without(...)` / `replace_rule(...)`
3. Sanity-check with `show_design_space(...)`
4. `ms.tune_tir(..., space=PostOrderApply(sch_rules=rules, postprocs="from-target", mutator_probs="from-target"))`

Don't touch `postprocs` / `mutator_probs` or the `structure` string — keeps the
independent variable clean. See `PLAN.md` for which experiments to run.